In [33]:
# ============================================================
# HOSPITALES DE GAZA: ATAQUES Y NIVEL DE AFECTACIÓN
# Visualización interactiva con Python + Altair
# ============================================================

# Instalar previamente:
# !pip install -q pandas altair vl-convert-python

import pandas as pd
import altair as alt
from urllib.parse import quote

# ------------------------------------------------------------
# 1. CARGAR Y LIMPIAR LOS DATOS
# ------------------------------------------------------------

archivo = "ataques_hospitales_database_limpia_2.0.csv"

# El archivo utiliza punto y coma como separador
df = pd.read_csv(archivo, sep=";")

# Limpiar espacios
for columna in ["Type of attack", "Category",
                "Hospital name", "Governorate", "Date"]:
    df[columna] = df[columna].astype(str).str.strip()

# Convertir fechas
df["Date_dt"] = pd.to_datetime(
    df["Date"], format="%d-%m-%Y", errors="coerce"
)

df = df.dropna(subset=[
    "Hospital name", "Governorate", "Date_dt"
])

df["Date_label"] = df["Date_dt"].dt.strftime("%d-%m-%Y")

# ------------------------------------------------------------
# 2. CONTABILIZAR ATAQUES POR HOSPITAL
# ------------------------------------------------------------

resumen = (
    df.groupby(["Hospital name", "Governorate"])
      .size()
      .reset_index(name="Cantidad_ataques")
)

# Ordenar hospitales según cantidad de ataques
resumen = resumen.sort_values(
    "Cantidad_ataques", ascending=False
).reset_index(drop=True)

# Calcular niveles visuales de afectación
q1, q2, q3 = resumen["Cantidad_ataques"].quantile(
    [0.25, 0.50, 0.75]
)

def nivel_afectacion(cantidad):
    if cantidad <= q1:
        return 0
    elif cantidad <= q2:
        return 1
    elif cantidad <= q3:
        return 2
    else:
        return 3

resumen["Nivel"] = resumen["Cantidad_ataques"].apply(
    nivel_afectacion
)

# ------------------------------------------------------------
# 3. CREAR ICONOS SVG DE HOSPITALES CON DISTINTOS DAÑOS
# ------------------------------------------------------------

def crear_icono(nivel):
    # Edificio base
    svg = """<svg xmlns="http://www.w3.org/2000/svg" width="120" height="120" viewBox="0 0 120 120">
      <path d="M25 105 V35 H95 V105 Z" fill="#e8e8e8" stroke="#333" stroke-width="3"/>
      <path d="M15 35 L60 12 L105 35" fill="#cccccc" stroke="#333" stroke-width="3"/>
      <rect x="48" y="65" width="24" height="40" fill="#aaaaaa" stroke="#333" stroke-width="2"/>
      <rect x="34" y="45" width="14" height="14" fill="#ffffff" stroke="#333" stroke-width="2"/>
      <rect x="72" y="45" width="14" height="14" fill="#ffffff" stroke="#333" stroke-width="2"/>
      <path d="M53 23 H67 M60 16 V30" stroke="#b22222" stroke-width="4"/>"""

    # Nivel 0: Hospital con menor cantidad de ataques
    if nivel == 0:
        pass

    # Nivel 1: Primeras grietas
    elif nivel == 1:
        svg += """<path d="M88 35 L82 48 L89 57" fill="none" stroke="#555" stroke-width="3"/>
        <path d="M25 80 L34 85" fill="none" stroke="#555" stroke-width="2"/>"""

    # Nivel 2: Daños visibles
    elif nivel == 2:
        svg += """<path d="M88 35 L78 48 L86 60 L77 72" fill="none" stroke="#333" stroke-width="4"/>
        <path d="M25 65 L37 74 L28 85" fill="none" stroke="#333" stroke-width="3"/>
        <rect x="72" y="45" width="14" height="14" fill="#777777"/>
        <path d="M15 105 L30 97 L42 105" fill="#999999"/>"""

    # Nivel 3: Mayor nivel de afectación registrada
    elif nivel == 3:
        svg += """<path d="M25 35 L40 48 L32 62 L45 75" fill="none" stroke="#222" stroke-width="4"/>
        <path d="M88 35 L76 50 L88 65 L73 83" fill="none" stroke="#222" stroke-width="4"/>
        <path d="M15 35 L35 24 L45 35" fill="#999999" stroke="#333" stroke-width="2"/>
        <rect x="34" y="45" width="14" height="14" fill="#777777"/>
        <rect x="72" y="45" width="14" height="14" fill="#777777"/>
        <path d="M12 105 L27 91 L42 105 L55 96 L68 105 L85 93 L105 105 Z" fill="#999999" stroke="#555" stroke-width="2"/>"""

    svg += "</svg>"
    return "data:image/svg+xml;charset=utf-8," + quote(svg)

resumen["icono"] = resumen["Nivel"].apply(crear_icono)

# ------------------------------------------------------------
# 4. DISTRIBUIR LOS HOSPITALES EN UNA CUADRÍCULA
# ------------------------------------------------------------

# Seis columnas para distribuir los hospitales
columnas = 6
resumen["columna"] = resumen.index % columnas
resumen["fila"] = resumen.index // columnas

# ------------------------------------------------------------
# 5. CREAR FILTRO INTERACTIVO POR GOBERNACIÓN
# ------------------------------------------------------------

gobernaciones = sorted(df["Governorate"].unique())

filtro_gobernacion = alt.param(
    name="Gobernacion",
    value="Todas",
    bind=alt.binding_select(
        options=["Todas"] + gobernaciones,
        name="Seleccionar gobernación: "
    )
)

# Selección interactiva de hospitales
seleccion_hospital = alt.selection_point(
    name="hospital_seleccionado",
    fields=["Hospital name"],
    on="click",
    clear="dblclick"
)

# Filtrar los hospitales según la gobernación elegida
base = alt.Chart(resumen).transform_filter(
    "Gobernacion == 'Todas' || datum.Governorate == Gobernacion"
).encode(
    x=alt.X(
        "columna:Q",
        scale=alt.Scale(domain=[-0.5, columnas - 0.5]),
        axis=None
    ),
    y=alt.Y(
        "fila:Q",
        scale=alt.Scale(
            domain=[-0.5, resumen["fila"].max() + 0.5],
            reverse=True
        ),
        axis=None
    )
)

# ------------------------------------------------------------
# 6. ICONOS DE HOSPITALES
# ------------------------------------------------------------

iconos = base.mark_image(
    width=75,
    height=75
).encode(
    url="icono:N",
    tooltip=[
        alt.Tooltip("Hospital name:N", title="Hospital"),
        alt.Tooltip("Governorate:N", title="Gobernación"),
        alt.Tooltip("Cantidad_ataques:Q", title="Ataques registrados"),
        alt.Tooltip("Nivel:Q", title="Nivel visual")
    ]
)

# ------------------------------------------------------------
# 7. NOMBRES DEBAJO DE LOS ICONOS
# ------------------------------------------------------------

nombres = base.mark_text(
    dy=48,
    fontSize=11,
    fontWeight="bold",
    align="center",
    baseline="top",
    limit=140
).encode(
    text="Hospital name:N"
)

# ------------------------------------------------------------
# 8. CAPA PARA SELECCIONAR LOS HOSPITALES
# ------------------------------------------------------------

clic_hospital = base.mark_point(
    opacity=0.001,
    size=7000
).encode(
    tooltip=[
        alt.Tooltip("Hospital name:N"),
        alt.Tooltip("Cantidad_ataques:Q")
    ]
).add_params(
    seleccion_hospital
)

# Unir las tres capas
grafico_hospitales = alt.layer(
    iconos,
    nombres,
    clic_hospital
).add_params(
    filtro_gobernacion
).properties(
    width=900,
    height=760,
    title=alt.Title(
        "Hospitales afectados en la Franja de Gaza",
        subtitle="Selecciona una gobernación y luego un hospital"
    )
)

# ------------------------------------------------------------
# 9. INFORMACIÓN DE LOS ATAQUES DEL HOSPITAL SELECCIONADO
# ------------------------------------------------------------

# Mostrar los 20 registros más recientes del hospital seleccionado.
datos_detalle = (
    alt.Chart(df)
    .transform_filter(seleccion_hospital)
    .transform_window(
        numero="row_number()",
        sort=[alt.SortField("Date_dt", order="descending")]
    )
    .transform_filter("datum.numero <= 20")
)

# Columna de fechas
fechas = datos_detalle.mark_text(
    align="left",
    fontSize=12
).encode(
    y=alt.Y(
        "numero:O",
        axis=None,
        sort="ascending"
    ),
    x=alt.value(5),
    text="Date_label:N"
).properties(
    width=110,
    height=500,
    title="Fecha"
)

# Columna de tipos de ataques
tipos = datos_detalle.mark_text(
    align="left",
    fontSize=12
).encode(
    y=alt.Y(
        "numero:O",
        axis=None,
        sort="ascending"
    ),
    x=alt.value(5),
    text="Type of attack:N",
    tooltip=[
        alt.Tooltip("Date_label:N", title="Fecha"),
        alt.Tooltip("Type of attack:N", title="Tipo de ataque"),
        alt.Tooltip("Category:N", title="Categoría"),
        alt.Tooltip("Closure:N", title="Fecha de cierre"),
        alt.Tooltip("Reopening:N", title="Reapertura")
    ]
).properties(
    width=480,
    height=500,
    title="Tipo de ataque registrado"
)

# Unir ambas columnas
detalle_ataques = alt.hconcat(
    fechas,
    tipos,
    spacing=15
).properties(
    title="REGISTROS DEL HOSPITAL SELECCIONADO"
)

# ------------------------------------------------------------
# 10. CONSTRUIR EL PANEL INTERACTIVO
# ------------------------------------------------------------

dashboard = alt.vconcat(
    grafico_hospitales,
    detalle_ataques,
    spacing=30
).resolve_scale(
    color="independent"
).configure_view(
    strokeWidth=0
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
).configure_title(
    fontSize=18,
    anchor="start"
).configure_concat(
    spacing=15
)

# ------------------------------------------------------------
# 11. MOSTRAR Y GUARDAR
# ------------------------------------------------------------

dashboard.display()

# Exportar como HTML interactivo
dashboard.save("hospitales_gaza_interactivo.html")
print("Visualización creada correctamente.")

alt.VConcatChart(...)

Visualización creada correctamente.
